# Fuzzy RI Pipeline with Fuzzy C-Means

This notebook contains the cleaned workflow used for the manuscript, extended with **Fuzzy C-Means (FCM)**.

## Contents
1. Load and inspect the dataset  
2. Check missing values  
3. Run the Elbow method  
4. Perform K-means clustering  
5. Calculate feature importance with Extra Trees  
6. Run Fuzzy C-Means with 6 clusters  
7. Export each individual's membership degrees to Excel  

**Dataset columns used:** `fT4`, `Sex`, `Age`, `TSH`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder

## 1. Data loading and basic checks

Update the file path if needed before running the notebook.


In [ ]:
DATA_PATH = "Reference_sample_group.xlsx"

# Main dataframe used throughout the analysis
df = pd.read_excel(DATA_PATH)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

## 2. Select the variables used for clustering

According to the dataset, the core variable names are written exactly as:

- `fT4`
- `Sex`
- `Age`
- `TSH`


In [ ]:
feature_columns = ["fT4", "Sex", "Age", "TSH"]

print("\nMissing values in core variables:")
print(df[feature_columns].isnull().sum())

# Keep only complete cases for clustering and FCM
df_model = df[feature_columns].dropna().copy()
X = df_model[feature_columns].astype(float).to_numpy()

print("\nClustering input preview:")
print(df_model.head())
print("\nModeling data shape:", X.shape)

## 3. Elbow method

This step helps suggest the optimal number of clusters.


In [ ]:
distortions = []
cluster_range = range(1, 20)

for k in cluster_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    distortions.append(model.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(cluster_range, distortions, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.tight_layout()
plt.show()

## 4. K-means clustering

The manuscript reports **6 clusters**, so the notebook uses `n_clusters = 6`.


In [ ]:
n_clusters = 6

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_model["kmeans"] = kmeans.fit_predict(X)

print("\nCluster counts:")
print(df_model["kmeans"].value_counts().sort_index())

print("\nData with cluster labels:")
print(df_model.head())

print("\nCluster-wise mean values:")
print(df_model.groupby("kmeans")[feature_columns].mean())

## 5. Feature importance with Extra Trees Classifier

The K-means cluster labels are used as the target variable in this section.


In [ ]:
X_et = df_model[feature_columns].astype(float).copy()
y_et = df_model["kmeans"].astype(int)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_et)

extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    criterion="entropy",
    max_features=2,
    random_state=42,
    n_jobs=-1
)

extra_trees.fit(X_et, y_encoded)

feature_importance = extra_trees.feature_importances_
feature_importance_std = np.std(
    [tree.feature_importances_ for tree in extra_trees.estimators_],
    axis=0
)

importance_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": feature_importance,
        "std": feature_importance_std,
    }
).sort_values("importance", ascending=False)

print("\nFeature importance results:")
print(importance_df)

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(
    importance_df["feature"],
    importance_df["importance"],
    yerr=importance_df["std"],
    capsize=5
)
plt.xlabel("Features")
plt.ylabel("Feature importance")
plt.title("Extra Trees Feature Importance")
plt.tight_layout()
plt.show()

## 6. Fuzzy C-Means (6 clusters)

This implementation is written directly in Python, so you do not need an extra package such as `scikit-fuzzy`.


In [ ]:
def initialize_membership_from_kmeans(X, kmeans_labels, n_clusters):
    n_samples = X.shape[0]
    U = np.zeros((n_clusters, n_samples), dtype=float)
    for i, label in enumerate(kmeans_labels):
        U[label, i] = 1.0

    eps = 1e-6
    U = U + eps
    U = U / U.sum(axis=0, keepdims=True)
    return U


def update_centers(X, U, m):
    um = U ** m
    centers = (um @ X) / um.sum(axis=1, keepdims=True)
    return centers


def calculate_distances(X, centers):
    distances = np.linalg.norm(X[None, :, :] - centers[:, None, :], axis=2)
    distances = np.fmax(distances, 1e-10)
    return distances


def update_membership(distances, m):
    power = 2.0 / (m - 1.0)
    ratio = (distances[:, None, :] / distances[None, :, :]) ** power
    U_new = 1.0 / ratio.sum(axis=1)
    return U_new


def fuzzy_c_means(
    X,
    n_clusters=6,
    m=2.0,
    max_iter=300,
    error=1e-5,
    random_state=42,
    init_labels=None,
):
    np.random.seed(random_state)
    n_samples = X.shape[0]

    if init_labels is not None:
        U = initialize_membership_from_kmeans(X, init_labels, n_clusters)
    else:
        U = np.random.rand(n_clusters, n_samples)
        U = U / U.sum(axis=0, keepdims=True)

    for iteration in range(max_iter):
        U_old = U.copy()
        centers = update_centers(X, U, m)
        distances = calculate_distances(X, centers)
        U = update_membership(distances, m)

        if np.linalg.norm(U - U_old) < error:
            print(f"FCM converged in {iteration + 1} iterations.")
            break
    else:
        print("FCM reached max_iter before full convergence.")

    centers = update_centers(X, U, m)
    distances = calculate_distances(X, centers)
    fuzzy_labels = np.argmax(U, axis=0)

    return centers, U, distances, fuzzy_labels

In [ ]:
fcm_centers, U, fcm_distances, fuzzy_labels = fuzzy_c_means(
    X,
    n_clusters=6,
    m=2.0,
    max_iter=300,
    error=1e-5,
    random_state=42,
    init_labels=df_model["kmeans"].values,
)

df_model["fcm_label"] = fuzzy_labels

for i in range(n_clusters):
    df_model[f"membership_cluster_{i+1}"] = U[i, :]

membership_columns = [f"membership_cluster_{i+1}" for i in range(n_clusters)]

print("\nFCM centers:")
print(pd.DataFrame(fcm_centers, columns=feature_columns))

print("\nFirst 5 rows with membership degrees:")
print(df_model[["kmeans", "fcm_label"] + membership_columns].head())

df_model["membership_sum"] = df_model[membership_columns].sum(axis=1)
print("\nMembership sum check (first 5 rows):")
print(df_model["membership_sum"].head())

In [ ]:
avg_membership_by_age = (
    df_model.groupby("Age")[membership_columns]
    .mean()
    .reset_index()
    .sort_values("Age")
)

plt.figure(figsize=(12, 7))
for col in membership_columns:
    plt.plot(avg_membership_by_age["Age"], avg_membership_by_age[col], label=col)
plt.xlabel("Age")
plt.ylabel("Average membership degree")
plt.title("Average FCM Membership Degrees by Age")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Export results to Excel

This file includes:
- `kmeans`
- `fcm_label`
- `membership_cluster_1` to `membership_cluster_6`
- `membership_sum`


In [ ]:
output_file = "reference_sample_group_with_fcm_memberships.xlsx"
df_model.to_excel(output_file, index=False)
print(f"Excel file created: {output_file}")

In [ ]:
# Uncomment these lines in Google Colab if you want the file to download directly.
# from google.colab import files
# files.download(output_file)

## Next step

The next step can be adding:
- individualized lower and upper reference limit calculations
- age-specific fuzzy reference intervals
- a second export file containing RI outputs
